# **Rastreando um pokémon em pseudovídeos**

## **Preparação das imagens**

Primeiramente, devemos importar bibliotecas com funcionalidades úteis ou que que pré implementam partes importantes para o programa.

In [ ]:
import cv2 as cv
import numpy as np

from IPython.display import Video, display

Conectando com o Drive para acessar os arquivos (esse passo pode ser desnecessário em alguns casos):

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Salvando os caminhos para os arquivos:

In [ ]:
VIDEO_PATH = '/content/drive/MyDrive/IC/labs_profis/videos/pseudovideo_1.mp4'
TEMPLATE_IMG = '/content/drive/MyDrive/IC/labs_profis/images/pokemon.png'  # recorte do objeto a rastrear
OUT_VIDEO = '/content/tracked_output.mp4'

###**Por que converter para escala de cinza?**

Como estamos usando uma técnica clássica de visão computacional, devemos converter a imagem do pokémon (o nosso template a ser encontrado, nesse caso) para escala de cinza. Isso é importante para simplificar os cálculos pois em uma imagem RGB, por exemplo, cada pixel tem 3 canais, enquanto em grayscale ela tem apenas um. Nesse caso, há apenas um canal em que realizar comparação entre a intensidade dos pixels da imagem de fundo e do template. Uma outra vantagem é que dessa forma a correlação passa a depender apenas da forma e do padrão dos elementos ao invés de suas cores.

Carregando o template do pokémon a ser encontrado em tons de cinza:

In [ ]:
template = cv.imread(TEMPLATE_IMG, cv.IMREAD_GRAYSCALE) # imagem vira uma matriz 2D do NumPy
th, tw = template.shape[::-1] # invertemos a imagem pois OpenCV espera (w,l) e NumPy retorna (h,w)

###**Tratando o vídeo**

Carregando o vídeo:

In [ ]:
cap = cv.VideoCapture(VIDEO_PATH)

fps = cap.get(cv.CAP_PROP_FPS) # carrega quantidade de frames por segundo

if fps is None or fps <= 0 or np.isnan(fps):
    fps = 30.0  # valor padrão seguro para a quantidade de fps

delay = 1.0/float(fps)

Abaixo há um trecho que cria o código do codec de compressão que será usado para salvar o vídeo. Basicamente, nós queremos codificar o vídeo a fim de comprimi-lo, para depois decodificá-lo e recuperar as informações originais. O cv.VideoWriter_fourcc converte a string em um código numérico que o OpenCV entende. Chamando writer.write(frame) o frame atual é adicionado ao vídeo de saída.

In [ ]:
width  = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))

fourcc = cv.VideoWriter_fourcc(*"mp4v")
writer = cv.VideoWriter(OUT_VIDEO, fourcc, fps, (width, height))

##**Performando o rastreamento pokémon**

Agora vamos iterar sobre os frames com o pokémon se movendo pelo cenário, sobrepondo janelas (patches) w x h do fundo sobre o template e comparando o resultado conforme o método escolhido.

In [ ]:
while True:
    # lendo os frames do vídeo
    ok, frame = cap.read()
    if not ok:  # acabou o vídeo
        break

    gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY) # convertendo o frame do fundo em grayscale

    # template matching (maior = melhor para TM_CCOEFF_NORMED)
    res = cv.matchTemplate(gray, template, cv.TM_CCOEFF_NORMED)
    _, max_val, _, max_loc = cv.minMaxLoc(res)

    top_left = max_loc
    bottom_right = (top_left[0] + tw, top_left[1] + th)

    cv.rectangle(frame, top_left, bottom_right, (0, 0, 255), 2)

    writer.write(frame)


cap.release()
writer.release()

###**Como funciona o template matching (dentre eles o cv.TM_CCOEFF_NORMED)?**

Trata-se do coeficiente de correlação normalizado entre o template e cada janela da imagem do fundo. A saída R(x,y) é um valor em [-1,1] tal que 1 é um match perfeito, 0 indica nenhuma correlação e -1 aparece quando os valores são opostos. A fórmula para calcular o R(x,y) pode ser consultada em: [OpenCV: Object Detection](https://docs.opencv.org/4.x/df/dfb/group__imgproc__object.html#gga3a7850640f1fe1f58fe91a2d7583695daf9c3ab9296f597ea71f056399a5831da).

## **Exibindo o vídeo do resultado no Colab**
Como o cv.imshow não funciona aqui, vamos precisar usar um método alternativo para visualizar o vídeo rastreado sem precisar baixá-lo no computador.

Para isso, vamos convertê-lo para um tipo de arquivo compatível com o Colab usando a biblioteca ```imageio``` e depois exibir o vídeo com o auxilio de outra biblioteca: ```IPython```.



In [ ]:
import imageio.v2 as imageio

reader = imageio.get_reader("/content/tracked_output.mp4")
writer = imageio.get_writer("/content/tracked_output.mp4", fps=reader.get_meta_data().get("fps", 30), codec="libx264", quality=8)

for frame in reader:
  writer.append_data(frame)

writer.close()

In [ ]:
from IPython.display import Video, display

VIDEO_PATH = "/content/tracked_output.mp4"  # ajuste o caminho
display(Video(VIDEO_PATH, embed=True, html_attributes="controls autoplay loop"))